# **EVAL AND EXPERIMENTS**

## **0. NEEDED IMPORTS**

In [1]:
# Basic Imports
import os
import sys
import json
import torch
import numpy as np 
import pandas as pd
import seaborn as sb
from pathlib import Path
import matplotlib.pyplot as plt


In [ ]:
# Run only one time
# 1. Get the base path
root = Path.cwd().parents[1]
# 2. Set that path as cwd
os.chdir(root)
# 3. Debug checking
print(f'·Aspect of the cwd:({Path.cwd()})')


·Aspect of the cwd:(/home/carlo/cuarto/GPT2_MOE/src)


In [ ]:
# File system imports
from gpt2_moe.model import GPTMOE
from gpt2_moe.train_test.utils import Utils
from gpt2_moe.train_test.configs import ExperimentConfig

CONFIG BASE DIR: /home/carlo/cuarto/GPT2_MOE/src/gpt2_moe/train_test/configs/cf_files


## **1. HISTORY AND MODEL LOADING**

In [ ]:
def path_checker(path:str, create:bool = False)->None:
    """
        Function aimed to check path existance
        and in case it does not exist and 
        wanted, create it 

        Params:
            path (str):
                The path to check 

            create(bool):
                The creation flag

        Returns:    
            None:
                No value is returned
    
    """

    # Check existance
    existance = os.path.exists(path)  
    if not existance:
        # Info
        print(f'(path) Path ({path}) does not exist')
        # Check if creation is seeked
        if create:
            print(' ·Path created')
            os.makedirs(path)  
        return None
    print(f'(path) Path ({path}) exists')

In [30]:
# Define params
path_fake = 'images'
path_exist = 'gpt2_moe'

# Test the function 
# Does not exist
path_checker(path = path_fake)
# Does not exist -> creation
path_checker(path = path_fake, create = True)
# Created
path_checker(path = path_fake)

# Existed
path_checker(path = path_exist)


(path) Path (images) does not exist
(path) Path (images) does not exist
 ·Path created
(path) Path (images) exists
(path) Path (gpt2_moe) exists


In [4]:
def weight_evo_retrieval(exp_name:str, path:str = 'gpt2_moe/checkpoints', seed:str = 'seed_42', model_name:str = 'full')->tuple[object, list[dict]]:
    """
        Function aimed to retrieve model weights 
        and its evo from a  given a source path

        Params:
            exp_name(str):
                The name of the experiment
            path(str):
                The source path
                Default value is gpt2_moe/checkpoints
            seed(str):
                The used to train the model
                Deafult value is 'seed_42'
            model_name(str):
                The name of the model to load
                Default name is 'full'
            

        Returns:
            A tuple of:
                - state_dict(collections.OrderedDict): Returns the state dict of the model 
                - history(list(dict)): Model's train evo 
            
    """

    # Check path existance
    if not os.path.exists(path):
        print(f'(w_loader) Debug :{' dir:'.join(list(os.scandir()))}')
        return {},{}

    # Define base full_path
    base_full_path = os.path.join(path, exp_name, seed, model_name)

    # Build the full path
    model_path = base_full_path + '.pth'
    print(f'(w_loader) Aspect of the full path ({model_path})')

    # Load the state_dict -> ensure no trojan is loaded :)
    state_dict = torch.load(model_path, weights_only = True)
    print(f'(w_loader) Weights loaded successfuly ({type(state_dict)})')

    # Load the evo 
    evo_path = base_full_path + '_history.json'
    with open(evo_path, mode = 'r') as file:
        evo = json.load(file)

    print(f'(w_loader) Evo loaded successfuly ({type(evo)})')

    return state_dict, evo

In [5]:
# Define mock params
path = 'gpt2_moe/checkpoints/'
exp_name = 'E2_scratch_moe_alternate'
seed = 'seed_42'

# Call the function
state_dict, evo = weight_evo_retrieval(path =  path, exp_name = exp_name, seed = seed)

(w_loader) Aspect of the full path (gpt2_moe/checkpoints/E2_scratch_moe_alternate/seed_42/full.pth)
(w_loader) Weights loaded successfuly (<class 'collections.OrderedDict'>)
(w_loader) Evo loaded successfuly (<class 'list'>)


In [6]:
def model_loader(param_source_name:str, target_model:dict)->tuple[GPTMOE, list[dict]]:
    """
        Function aimed to load a model and its evo given
        a weight path and a param definition 
        path.


        Params:
            param_source_name(str):
                YAML name where params live
            target_model(dict):
                Dictioary containing needed data
                to retrieve the seeked model

        Returns:
            A tuple of:
                - model(GPTMOE): The initialized model 
                - evo (list(dict)): Model's training evo
    """

    # Get the params
    config = ExperimentConfig(name = param_source_name).get_model_params()
    
    # Initialize the model 
    model = GPTMOE(config)

    # Recover state_dict
    state_dict, evo = weight_evo_retrieval(**target_model)

    # Load the weights
    model.load_state_dict(state_dict)

    return model, evo

In [15]:
def experts_loader(param_source_name:str, base_name:str, target_model:dict, experts:int)->list[tuple[object, list[dict]]]:
    """
        Function acting as a wrapper for
        model_loader to enable experts 
        loading

        Params:
            param_source_name(str):
                YAML name where params live
            base_name(str):
                The expert base name
            target_model(dict):
                Dictionary containing needed data
                to retrieve the seeked model
            experts(int): 
                Amount of experts to load

        Returns:
            A list of tuples:
                - model(GPTMOE): The initialized model 
                - evo (list(dict)): Model's training evo
            
    """
    # Define the storage
    storage = []
    # Load each expert
    for i in range(experts):
        print(f'(expert_load)Loading expert {i}')
        # Define  expert name
        expert_name = base_name.format(i)
        print(f'    Selected expert:{expert_name}')
        # Update its name for loading
        target_model["model_name"]  = expert_name
        print(f'    Updated dict:{expert_name}')
        # Load weights and evo
        storage.append(model_loader(param_source_name = param_source_name, target_model = target_model))
        print()
    return storage


### **1.1.1 DENSE MODEL LOADING**

#### **FULL MODEL**

In [12]:
# Define the parameters
param_source_name = 'test_1.yaml'
target_model = {"exp_name": "E1_scratch_dense"}

# Load the model 
dense_model, evo_dense = model_loader(param_source_name = param_source_name, target_model = target_model)


(w_loader) Aspect of the full path (gpt2_moe/checkpoints/E1_scratch_dense/seed_42/full.pth)
(w_loader) Weights loaded successfuly (<class 'collections.OrderedDict'>)
(w_loader) Evo loaded successfuly (<class 'list'>)


### **1.1.2 ALTERNATE MODEL LOADING**

#### **LOAD THE INDIVIDUAL EXPERTS**

In [16]:
# Define the parameters
base_name = 'expert_{}'
param_source_name = 'test_2.yaml'
target_model = {"exp_name": "E2_scratch_moe_alternate"}

# Define the storage
n_experts = 6

# Load the experts 
experts_storage = experts_loader(param_source_name = param_source_name, base_name = base_name,  target_model = target_model, experts = 4)


(expert_load)Loading expert 0
    Selected expert:expert_0
    Updated dict:expert_0
(w_loader) Aspect of the full path (gpt2_moe/checkpoints/E2_scratch_moe_alternate/seed_42/expert_0.pth)
(w_loader) Weights loaded successfuly (<class 'collections.OrderedDict'>)
(w_loader) Evo loaded successfuly (<class 'list'>)

(expert_load)Loading expert 1
    Selected expert:expert_1
    Updated dict:expert_1
(w_loader) Aspect of the full path (gpt2_moe/checkpoints/E2_scratch_moe_alternate/seed_42/expert_1.pth)
(w_loader) Weights loaded successfuly (<class 'collections.OrderedDict'>)
(w_loader) Evo loaded successfuly (<class 'list'>)

(expert_load)Loading expert 2
    Selected expert:expert_2
    Updated dict:expert_2
(w_loader) Aspect of the full path (gpt2_moe/checkpoints/E2_scratch_moe_alternate/seed_42/expert_2.pth)
(w_loader) Weights loaded successfuly (<class 'collections.OrderedDict'>)
(w_loader) Evo loaded successfuly (<class 'list'>)

(expert_load)Loading expert 3
    Selected expert:exper

#### **LOAD THE FULL MODEL**

In [20]:
# Define the parameters
param_source_name = 'test_2.yaml'
target_model = {"exp_name": "E2_scratch_moe_alternate"}

# Load the model 
full_alternate, full_alternate_evo = model_loader(param_source_name = param_source_name, target_model = target_model)


(w_loader) Aspect of the full path (gpt2_moe/checkpoints/E2_scratch_moe_alternate/seed_42/full.pth)
(w_loader) Weights loaded successfuly (<class 'collections.OrderedDict'>)
(w_loader) Evo loaded successfuly (<class 'list'>)


## **2. TRAIN STUDY**

## **3. TEST QUALITY EVAL**

## **4. HYPOTHESIS RELATED EXPERIMENTS**